<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Algopro/Algopro%20MultiHeadAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!gdown 1d7qwMrVyLzc7S-45twjS5iZMe9d5BYyu

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import random
from datetime import datetime, timedelta

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import json
from collections import Counter

DATA_PATH = "dual_stream.jsonl"

data = []

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = obj["text"]
        target = obj["target"]

        data.append((text, target))

random.shuffle(data)

print(f"Loaded {len(data)} examples")

print(f"\nSample (input -> target) pairs:")
for inp, target in data[:5]:
    preview = inp if len(inp) <= 150 else inp[:147] + "..."
    print(f"  {preview}\n    -> {target}\n")

lengths = [len(inp) for inp, _ in data]
print(f"Input length: min: {min(lengths)}, avg: {sum(lengths)/len(lengths):.1f}, max: {max(lengths)}")

In [ ]:
from collections import Counter
input_char_counts = Counter()
output_char_counts = Counter()
for inp, target in data:
    input_char_counts.update(inp)
    output_char_counts.update(target)

# keep all chars that appear at least 2 times
input_chars = sorted(c for c, n in input_char_counts.items() if n >= 2)
output_chars = sorted(output_char_counts.keys())

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

input_vocab = [PAD, UNK] + input_chars
output_vocab = [PAD, SOS, EOS] + output_chars

input_stoi = {c: i for i, c in enumerate(input_vocab)}
input_itos = {i: c for i, c in enumerate(input_vocab)}
output_stoi = {c: i for i, c in enumerate(output_vocab)}
output_itos = {i: c for i, c in enumerate(output_vocab)}

print(f"Input vocab size:  {len(input_vocab)}")
print(f"Output vocab size: {len(output_vocab)}")

# cap input length at 99th percentile to avoid outliers bloating padding
sorted_lens = sorted(len(inp) for inp, _ in data)
MAX_INPUT_LEN = sorted_lens[int(0.99 * len(sorted_lens))]
MAX_OUTPUT_LEN = max(len(tgt) for _, tgt in data) + 2  # +SOS +EOS
print(f"Max input length (99th pct): {MAX_INPUT_LEN}")
print(f"Max output length: {MAX_OUTPUT_LEN}")

# drop examples above the cap so we don't truncate actual dates
kept = [(inp, tgt) for inp, tgt in data if len(inp) <= MAX_INPUT_LEN]
print(f"Keeping {len(kept)}/{len(data)} examples after length cap")
data = kept

def encode_input(s):
    ids = [input_stoi.get(c, input_stoi[UNK]) for c in s]
    ids += [input_stoi[PAD]] * (MAX_INPUT_LEN - len(ids))
    return ids

def encode_output(s):
    ids = [output_stoi[SOS]] + [output_stoi[c] for c in s] + [output_stoi[EOS]]
    ids += [output_stoi[PAD]] * (MAX_OUTPUT_LEN - len(ids))
    return ids

class DateDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        inp, tgt = self.pairs[idx]
        return (torch.tensor(encode_input(inp), dtype=torch.long),
                torch.tensor(encode_output(tgt), dtype=torch.long),
                inp, tgt)

split = int(0.9 * len(data))
train_data = data[:split]
test_data = data[split:]

train_loader = DataLoader(DateDataset(train_data), batch_size=64, shuffle=True,
                          collate_fn=lambda b: (torch.stack([x[0] for x in b]),
                                                torch.stack([x[1] for x in b]),
                                                [x[2] for x in b],
                                                [x[3] for x in b]))
test_loader = DataLoader(DateDataset(test_data), batch_size=64,
                         collate_fn=lambda b: (torch.stack([x[0] for x in b]),
                                               torch.stack([x[1] for x in b]),
                                               [x[2] for x in b],
                                               [x[3] for x in b]))

print(f"\nTrain size: {len(train_data)}, Test size: {len(test_data)}")

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (batch, n_queries, d_k)       - queries
    K: (batch, n_keys, d_k)          - keys
    V: (batch, n_keys, d_v)          - values
    mask: (batch, n_queries, n_keys) - 1 where valid, 0 where padded

    Returns:
      output:     (batch, n_queries, d_v)
      attn_weights: (batch, n_queries, n_keys)
    """
    d_k = Q.size(-1)

    # step 1: compute raw scores Q @ K^T
    # shape: (batch, n_queries, n_keys)
    scores = ...

    # step 2: scale by sqrt(d_k) for softmax


    # step 3: apply mask (set padded positions to -inf so softmax ignores them)
    if mask is not None:
        ...

    # step 4: softmax over the keys dimension -> attention weights
    attn_weights = ...

    # step 5: weighted sum of values
    output = ...

    return output, attn_weights


# check on tiny inputs
Q_test = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]]) # 2 queries
K_test = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]) # 3 keys
V_test = torch.tensor([[[10.0, 0.0], [0.0, 20.0], [5.0, 5.0]]]) # 3 values

out, w = scaled_dot_product_attention(Q_test, K_test, V_test)
print(f"Attention weights shape: {w.shape}")
print(f"Attention weights:\n{w}")
print(f"Output:\n{out}")

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = ...
        self.lstm = ...
        self.proj = ... # merge bidirectional

    def forward(self, x):
        ...

class MultiheadAttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_heads=2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads

        self.embedding = ...
        self.lstm_cell = ...

        # project to (hidden_dim * num_heads) to generate Q, K, V for all heads at once.
        self.W_q = ...
        self.W_k = ...
        self.W_v = ...

        # MHA requires a final output projection to mix the concatenated heads back to hidden_dim
        self.W_o = ...

        self.out_proj = ...

    def forward(self, encoder_outputs, input_mask, target=None, max_len=MAX_OUTPUT_LEN):
        """
        encoder_outputs: (B, T_in, hidden_dim)
        input_mask: (B, T_in) - 1 for real tokens, 0 for padding
        target: (B, T_out) - ground-truth output for teacher forcing (training)
        """
        B, T_in, H = encoder_outputs.shape

        # pre-compute K and V from encoder outputs
        # Shape: (B, T_in, H * num_heads) -> (B, T_in, num_heads, H) -> (B, num_heads, T_in, H)


        # initialize decoder hidden state


        # first input to decoder is SOS
        prev_token = ...

        logits_list = []
        attn_list = []

        T_out = target.size(1) - 1 if target is not None else max_len - 1

        # build mask for attention: (B, 1, 1, T_in), broadcast over heads and the single query
        attn_mask = ...

        for t in range(T_out):
            # embed previous token


            # compute query from current decoder hidden state
            # Shape: (B, H) -> (B, 1, H * num_heads) -> (B, 1, num_heads, H) -> (B, num_heads, 1, H)


            # attention over encoder outputs


            # Concatenate the heads back together
            # context is (B, num_heads, 1, H) -> transpose to (B, 1, num_heads, H)
            # -> flatten back to (B, 1, num_heads * H)


            # project the concatenated heads back down to the single hidden_dim


            # append to attn_list, attn_w is (B, num_heads, 1, T_in) -> (B, num_heads, T_in)


            # feed [embedding, context] into the LSTM cell


            # produce output logits using both hidden state and context


            # append logits to logits list


            # pick next input token (next token from target if it exists, greedy otherwise)


        logits_stack = ...
        attn_stack = ...
        return logits_stack, attn_stack


class Seq2SeqMultiHeadAttn(nn.Module):
    def __init__(self, input_vocab_size, output_vocab_size,
                 embed_dim=32, hidden_dim=64, num_heads=2):
        super().__init__()
        self.encoder = Encoder(input_vocab_size, embed_dim, hidden_dim)
        self.decoder = MultiheadAttentionDecoder(output_vocab_size, embed_dim, hidden_dim, num_heads)

    def forward(self, src, target=None):
        input_mask = ...
        enc_out = ...
        logits, attn = ...
        return logits, attn


model = Seq2SeqMultiHeadAttn(len(input_vocab), len(output_vocab), num_heads=2).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=output_stoi[PAD])

N_EPOCHS = 10

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for src, target, _, _ in train_loader:
        src, target = src.to(device), target.to(device)
        logits, _ = model(src, target)
        # targets for loss: shift by 1 (predict tokens from position 1 onward)
        ...

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for src, target, _, target_strings in test_loader:
            src = src.to(device)
            logits, _ = model(src, target=None) # no teacher forcing for inference
            preds = logits.argmax(-1).cpu().tolist()
            for pred_ids, true_str in zip(preds, target_strings):
                chars = []
                for i in pred_ids:
                    if i == output_stoi[EOS]:
                        break
                    if i == output_stoi[PAD]:
                        continue
                    chars.append(output_itos[i])
                pred_str = "".join(chars)
                if pred_str == true_str:
                    correct += 1
                total += 1

    print(f"Epoch {epoch}: train loss = {total_loss/n_batches:.4f} | "
          f"test exact-match = {correct}/{total} ({100*correct/total:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

text = input("Enter text: ")

model.eval()

encoded = encode_input(text)
input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)

with torch.no_grad():
    logits, attn = model(input_tensor, target=None)

pred_ids = logits.argmax(-1)[0].cpu().tolist()
chars = []
for i in pred_ids:
    if i == output_stoi[EOS]:
        break

    if i not in (output_stoi[PAD], output_stoi[SOS]):
        chars.append(output_itos[i])

output = "".join(chars)

print(f"\nPredicted Output: {output}")

attn_tensor = attn[0].cpu().numpy()

out_len = len(chars)
in_len = len(text)

num_heads = attn_tensor.shape[1]

# create a vertical stack of subplots for each head
fig, axes = plt.subplots(num_heads, 1, figsize=(max(10, in_len * 0.25), max(3, out_len * 0.3) * num_heads))

# if num_heads is 1, axes isn't an array, so we wrap it in a list for the loop
if num_heads == 1:
    axes = [axes]

for h in range(num_heads):
    # extract attention for this specific head and trim out the <pad> tokens
    head_attn = attn_tensor[:out_len, h, :in_len]

    ax = sns.heatmap(head_attn,
                xticklabels=list(text),
                yticklabels=list(output),
                cmap="viridis",
                cbar_kws={'label': 'Attention Weight'},
                ax=axes[h])

    # move the X-axis labels to the top
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')

    # keep labels flat for readability
    for tick in ax.get_xticklabels():
        tick.set_rotation(0)
    for tick in ax.get_yticklabels():
        tick.set_rotation(0)

    axes[h].set_title(f"Attention Head {h + 1}", pad=15, fontweight='bold', fontsize=14)
    axes[h].set_ylabel("Output Sequence")

axes[-1].set_xlabel("Input Sequence", labelpad=15, fontsize=12)
plt.tight_layout()
plt.show()